# Phase 8 — AI-Assisted Business Insights (Optional Enhancement)
**Project:** AI-Based Customer Segmentation and Purchasing Behaviour Analysis Using Machine Learning

## Purpose
This optional component turns already-aggregated customer-segment statistics into plain-language business insights. The core machine-learning project must continue to work without an external AI API.

## Privacy and reliability
- Send only aggregate cluster profiles, never customer-level rows, names, invoice IDs, or other identifying details.
- Keep API keys in environment variables or Colab Secrets; never paste keys into the notebook or commit them.
- Treat generated recommendations as suggestions to review, not verified facts or causal claims.
- If the API is unavailable or no key is configured, use the local fallback summary.
- This phase is optional; the assignment brief does not explicitly require a generative-AI API.

In [1]:
import os
from pathlib import Path
# Work inside data/processed (found from the notebook folder upwards); paths below are relative to it.
for _d in [Path.cwd(), *Path.cwd().parents]:
    if (_d / "data" / "processed").is_dir():
        os.chdir(_d / "data" / "processed")
        break
print("Data folder:", Path.cwd().name)
import pandas as pd

PROFILE_FILE = Path("Phase7_KMeans_Segment_Profiles.csv")
profiles = pd.read_csv(PROFILE_FILE)
required = ["KMeans_Cluster", "Customers", "Customer_Percent",
            "Recency_Median", "Frequency_Median", "Monetary_Median"]
missing = [c for c in required if c not in profiles.columns]
if missing:
    raise ValueError(f"Missing required profile columns: {missing}")

display(profiles[required].round(2))


Data folder: processed


,KMeans_Cluster,Customers,Customer_Percent,Recency_Median,Frequency_Median,Monetary_Median
0,0,2038,46.98,46.0,2.0,508.80
1,1,980,22.59,253.0,1.0,290.46
2,2,1320,30.43,18.0,7.0,2570.50


In [2]:
def local_business_summary(profiles):
    lines = [
        "Customer Segmentation — Business Insights",
        "These are descriptive suggestions based on historical RFM summaries, not causal conclusions."
    ]
    for _, r in profiles.iterrows():
        lines.append(
            f"Cluster {int(r['KMeans_Cluster'])} ({r.get('Suggested_Segment_Name', 'unnamed')}): {int(r['Customers'])} customers "
            f"({r['Customer_Percent']:.1f}%); median recency {r['Recency_Median']:.0f} days; "
            f"median frequency {r['Frequency_Median']:.0f} orders; "
            f"median monetary value £{r['Monetary_Median']:.2f}."
        )
        if r["Recency_Median"] > profiles["Recency_Median"].median() and r["Frequency_Median"] < profiles["Frequency_Median"].median():
            lines.append("Possible action to test: a measured re-engagement campaign, with a holdout group to assess impact.")
        elif r["Frequency_Median"] > profiles["Frequency_Median"].median() and r["Monetary_Median"] > profiles["Monetary_Median"].median():
            lines.append("Possible action to test: loyalty recognition or relevant cross-sell recommendations.")
        else:
            lines.append("Possible action to test: encourage repeat purchases with relevant product suggestions and track repeat-order rate.")
    return "\n".join(lines)

fallback_text = local_business_summary(profiles)
print(fallback_text)


Customer Segmentation — Business Insights
These are descriptive suggestions based on historical RFM summaries, not causal conclusions.
Cluster 0 (Occasional / Lower-Spend Customers): 2038 customers (47.0%); median recency 46 days; median frequency 2 orders; median monetary value £508.80.
Possible action to test: encourage repeat purchases with relevant product suggestions and track repeat-order rate.
Cluster 1 (At-Risk / Inactive, Low-Frequency Customers): 980 customers (22.6%); median recency 253 days; median frequency 1 orders; median monetary value £290.45.
Possible action to test: a measured re-engagement campaign, with a holdout group to assess impact.
Cluster 2 (Recent, Frequent, High-Spend Customers): 1320 customers (30.4%); median recency 18 days; median frequency 7 orders; median monetary value £2570.49.
Possible action to test: loyalty recognition or relevant cross-sell recommendations.


## Optional API integration
The following example uses the OpenAI API if the package and `OPENAI_API_KEY` environment variable are configured. It sends only aggregate profile values. If unavailable, the notebook keeps working with the local summary above.

Before running, install the SDK if needed (`pip install openai`) and set `OPENAI_API_KEY` using Colab Secrets or your local environment. Never place the key directly in code.

In [3]:
# Optional: uncomment and run only if you have configured OPENAI_API_KEY securely.
# This call is not needed to complete the core ML assignment.
#
# from openai import OpenAI
# import json
#
# api_key = os.getenv("OPENAI_API_KEY")
# if api_key:
#     client = OpenAI(api_key=api_key)
#     aggregate_payload = profiles[required].round(2).to_dict(orient="records")
#     response = client.responses.create(
#         model="gpt-4.1-mini",
#         input=[
#             {"role": "system", "content":
#              "You are a cautious retail analytics assistant. Explain aggregate RFM segments in plain language. "
#              "Only use provided values, label recommendations as hypotheses, do not claim causality, "
#              "and do not infer personal traits."},
#             {"role": "user", "content":
#              "Summarize these customer segment aggregates and suggest one measurable business experiment per segment. "
#              "Mention limitations. Data: " + json.dumps(aggregate_payload)}
#         ]
#     )
#     print(response.output_text)
# else:
#     print("OPENAI_API_KEY not configured; using the local summary instead.")


## Evaluation and limitations
- Check that the generated text accurately reflects the supplied aggregate values.
- Recommendations should be tested using measurable outcomes such as repeat-order rate or reactivation rate.
- This component does not improve or validate the clustering model itself.
- API model names and SDK interfaces may change; check current official documentation before enabling the optional API call.
- Do not include customer-level or identifying data in API requests.